# DermaScan — EfficientNet-B3 retraining (Kaggle GPU → CPU deployment)

Trains the skin-lesion classifier on HAM10000 and exports a **self-describing,
CPU-ready** artifact bundle that drops straight into the serving repository.

## What this notebook fixes

Each of these caused a real defect in the deployed system. They are designed in
here rather than patched afterwards.

| Past failure | How this notebook prevents it |
| :--- | :--- |
| Served checkpoint silently differed from the evaluated one (20-point accuracy gap) | The checkpoint carries its own `arch`, `head`, `classes` and a weight fingerprint. Serving verifies rather than assumes. |
| Inference ran at 224 px centre-crop while training used 300 px | `img_size` and the exact normalization are stored **in** the checkpoint; validation transforms are identical to serving transforms. |
| Thresholds fitted on softmax, served under sigmoid | `readout` is recorded and the thresholds are fitted under that same readout. |
| Confidence ~0.97 on wrong answers (ECE 0.116) | Temperature fitted on a held-out calibration split and exported. |
| Melanoma recall 0.624, misses reported as benign | Melanoma-weighted loss, plus an alert threshold fitted for a target sensitivity. |
| OOD gate biased against darker skin | Brightness/contrast augmentation, and an explicit brightness-robustness check before export. |
| `torch.load` failed on CPU due to numpy scalars in the checkpoint | Everything saved is a tensor or a plain Python primitive, so `weights_only=True` works. |
| Metrics reported that were never measured | The test split is touched exactly once, at the end, and never used for any fitting. |

## Before running

1. **Settings → Accelerator → GPU** (P100 or T4).
2. **Add data**: search for `skin-cancer-mnist-ham10000` and add it.
3. Run all. Roughly 2–4 h at 30 epochs; reduce `EPOCHS` for a smoke run.

In [3]:
!pip install -q timm torchvision

In [4]:
import os, sys, json, math, random, time
from collections import Counter
from datetime import datetime, timezone

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision.transforms as transforms
import timm

# 1. Reproducibility (Crucial for consistent training runs)
def seed_everything(seed=42):
    random.seed(seed)
    os.environ['PYTHONHASHSEED'] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(42)

# 2. Hardware Configuration
print(f"PyTorch Version: {torch.__version__} | Timm Version: {timm.__version__}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if device.type == "cuda":
    print(f"GPU Detected: {torch.cuda.get_device_name(0)}")
    # Re-enable benchmark for faster training since image sizes are mostly static
    torch.backends.cudnn.benchmark = True 
else:
    print("WARNING: No GPU detected. Settings -> Accelerator -> GPU.")

# 3. Model Architecture Initialization
# Configured for 7 classes (AKIEC, BCC, BKL, DF, MEL, NV, VASC)
num_classes = 7 
model = timm.create_model('efficientnet_b3', pretrained=True, num_classes=num_classes)
model = model.to(device)

print(f"Successfully loaded EfficientNet-B3 mapped to {num_classes} classes on {device}.")

PyTorch Version: 2.10.0+cu128 | Timm Version: 1.0.26
GPU Detected: Tesla T4


model.safetensors:   0%|          | 0.00/49.3M [00:00<?, ?B/s]

Successfully loaded EfficientNet-B3 mapped to 7 classes on cuda.


In [5]:
# ── Configuration ────────────────────────────────────────────────────────────
CFG = {
    "seed": 42,

    # Architecture. "plain" = timm head (matches the current serving default);
    # "multihead" = the two-layer head. Whichever is chosen is recorded in the
    # checkpoint, so serving can never guess wrong again.
    "arch": "efficientnet_b3",
    "head": "plain",
    "num_classes": 7,

    # Must match serving exactly. Stored in the checkpoint.
    "img_size": 300,
    "norm_mean": [0.485, 0.456, 0.406],
    "norm_std": [0.229, 0.224, 0.225],

    # Readout the thresholds will be fitted under, and that serving must use.
    "readout": "softmax",

    "epochs": 30,
    "batch_size": 32,
    "lr_head": 3e-4,
    "lr_backbone": 3e-5,
    "weight_decay": 1e-4,
    "warmup_epochs": 2,
    "label_smoothing": 0.05,
    "focal_gamma": 2.0,
    "drop_rate": 0.4,
    "drop_path_rate": 0.2,

    # Melanoma is the class whose errors matter most. This multiplies its loss
    # contribution on top of inverse-frequency weighting.
    "mel_loss_weight": 2.5,
    "target_mel_sensitivity": 0.90,

    "early_stop_patience": 8,
    "num_workers": 2,
    "amp": True,
}

CLASSES = ["akiec", "bcc", "bkl", "df", "mel", "nv", "vasc"]   # ORDER IS CONTRACT
MEL_IDX = CLASSES.index("mel")
OUT_DIR = "/kaggle/working"

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = False   # speed; determinism is seeded above
    torch.backends.cudnn.benchmark = True

seed_everything(CFG["seed"])
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(json.dumps(CFG, indent=2))

{
  "seed": 42,
  "arch": "efficientnet_b3",
  "head": "plain",
  "num_classes": 7,
  "img_size": 300,
  "norm_mean": [
    0.485,
    0.456,
    0.406
  ],
  "norm_std": [
    0.229,
    0.224,
    0.225
  ],
  "readout": "softmax",
  "epochs": 30,
  "batch_size": 32,
  "lr_head": 0.0003,
  "lr_backbone": 3e-05,
  "weight_decay": 0.0001,
  "warmup_epochs": 2,
  "label_smoothing": 0.05,
  "focal_gamma": 2.0,
  "drop_rate": 0.4,
  "drop_path_rate": 0.2,
  "mel_loss_weight": 2.5,
  "target_mel_sensitivity": 0.9,
  "early_stop_patience": 8,
  "num_workers": 2,
  "amp": true
}


## 1. Data

HAM10000 contains multiple images of the *same lesion*. Splitting by image leaks
the same lesion into train and test and inflates every metric. Splits below are
**grouped by `lesion_id`** and stratified by diagnosis.

In [8]:
import os
import pandas as pd

# ── 1. Locate the dataset dynamically ────────────────────────────────────────
ROOT = None

# Walk through every single directory inside /kaggle/input
for dirpath, _, files in os.walk('/kaggle/input'):
    if "HAM10000_metadata.csv" in files:
        ROOT = dirpath
        break

if ROOT is None:
    # If it still fails, the data isn't mounted at all. 
    raise SystemExit(f"Data not found. /kaggle/input/ contents: {os.listdir('/kaggle/input/')}. Please remove the dataset from the sidebar and add it again.")

print("dataset root:", ROOT)

# ── 2. Load Metadata ─────────────────────────────────────────────────────────
meta_path = os.path.join(ROOT, "HAM10000_metadata.csv")
meta = pd.read_csv(meta_path)
print("metadata rows:", len(meta))

# ── 3. Map Image Paths ───────────────────────────────────────────────────────
index = {}

# Search from the ROOT down to find all .jpg files (handles part_1/part_2 splits automatically)
for dirpath, _, files in os.walk(ROOT):
    for f in files:
        if f.lower().endswith(".jpg"):
            index[os.path.splitext(f)[0]] = os.path.join(dirpath, f)

meta["path"] = meta["image_id"].map(index)
missing = meta["path"].isna().sum()
meta = meta.dropna(subset=["path"]).reset_index(drop=True)

print(f"images located: {len(meta)} | missing: {missing}")
print(meta["dx"].value_counts().to_string())
print("\nDataset successfully mapped for EfficientNet-B3 pipeline.")

dataset root: /kaggle/input/datasets/kmader/skin-cancer-mnist-ham10000
metadata rows: 10015
images located: 10015 | missing: 0
dx
nv       6705
mel      1113
bkl      1099
bcc       514
akiec     327
vasc      142
df        115

Dataset successfully mapped for EfficientNet-B3 pipeline.


In [9]:
# ── Lesion-disjoint stratified splits ────────────────────────────────────────
# One lesion contributes all of its images to exactly one split.
def make_splits(df, seed, fractions=(0.60, 0.15, 0.10, 0.15)):
    rng = np.random.RandomState(seed)
    lesion_dx = df.groupby("lesion_id")["dx"].first()
    assign = {}
    names = ["train", "val", "calib", "test"]
    for dx, group in lesion_dx.groupby(lesion_dx):
        lesions = group.index.to_numpy()
        rng.shuffle(lesions)
        cuts = (np.cumsum(fractions) * len(lesions)).astype(int)
        for name, chunk in zip(names, np.split(lesions, cuts[:-1])):
            for les in chunk:
                assign[les] = name
    out = df.copy()
    out["split"] = out["lesion_id"].map(assign)
    return out

data = make_splits(meta, CFG["seed"])
print(pd.crosstab(data["split"], data["dx"], margins=True).to_string())

# Leakage assertions - cheap, and this is exactly the class of bug that silently
# inflates results.
for a in ["train", "val", "calib", "test"]:
    for b in ["train", "val", "calib", "test"]:
        if a >= b: continue
        overlap = set(data[data.split == a].lesion_id) & set(data[data.split == b].lesion_id)
        assert not overlap, f"lesion leakage between {a} and {b}: {len(overlap)}"
assert data["split"].notna().all()
print("\nno lesion appears in more than one split")

for name in ["train", "val", "calib", "test"]:
    data[data.split == name][["image_id", "lesion_id", "dx", "path"]].to_csv(
        f"{OUT_DIR}/split_{name}.csv", index=False)
print("split manifests written")

dx     akiec  bcc   bkl   df   mel    nv  vasc    All
split                                                
calib     34   49   109   15   110   678    14   1009
test      52   79   158   22   170  1000    22   1503
train    196  310   669   60   663  3992    85   5975
val       45   76   163   18   170  1035    21   1528
All      327  514  1099  115  1113  6705   142  10015

no lesion appears in more than one split
split manifests written


In [10]:
# ── Transforms ───────────────────────────────────────────────────────────────
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from torchvision import transforms
from PIL import Image

NORM = transforms.Normalize(mean=CFG["norm_mean"], std=CFG["norm_std"])
S = CFG["img_size"]

# Brightness and contrast jitter is not cosmetic: the deployed system was found
# to change its answer when an image was darkened, which biases against darker
# skin and poor lighting. Training across that range is the fix.
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(S, scale=(0.75, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomApply([transforms.RandomRotation(25)], p=0.5),
    transforms.ColorJitter(brightness=0.30, contrast=0.30,
                           saturation=0.20, hue=0.02),
    transforms.ToTensor(),
    NORM,
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.10)),
])

# IDENTICAL to serving. Any divergence here is the 224-vs-300 bug all over again.
eval_tf = transforms.Compose([
    transforms.Resize((S, S)),
    transforms.ToTensor(),
    NORM,
])

class LesionDataset(Dataset):
    def __init__(self, frame, transform):
        self.paths = frame["path"].tolist()
        self.labels = [CLASSES.index(d) for d in frame["dx"]]
        self.transform = transform
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        img = Image.open(self.paths[i]).convert("RGB")
        return self.transform(img), self.labels[i]

def loader_for(split, transform, shuffle=False, sampler=None):
    ds = LesionDataset(data[data.split == split], transform)
    return DataLoader(ds, batch_size=CFG["batch_size"], shuffle=shuffle,
                      sampler=sampler, num_workers=CFG["num_workers"],
                      pin_memory=True, drop_last=(sampler is not None))

# Balanced sampling: nv is ~67% of the corpus and dominates the gradient.
train_frame = data[data.split == "train"]
counts = Counter(train_frame["dx"])
weights = train_frame["dx"].map(lambda d: 1.0 / counts[d]).to_numpy()
sampler = WeightedRandomSampler(torch.DoubleTensor(weights), len(weights),
                                replacement=True)

train_loader = loader_for("train", train_tf, sampler=sampler)
val_loader   = loader_for("val", eval_tf)
calib_loader = loader_for("calib", eval_tf)
test_loader  = loader_for("test", eval_tf)
print({k: len(data[data.split == k]) for k in ["train", "val", "calib", "test"]})

{'train': 5975, 'val': 1528, 'calib': 1009, 'test': 1503}


## 2. Model and loss

The head choice is recorded in the checkpoint. `plain` matches the serving
default; `multihead` reproduces the two-layer head.

In [11]:
# ── Model ────────────────────────────────────────────────────────────────────
class ClassifierHead(nn.Module):
    '''Two-layer head with independent dropout gates.'''
    def __init__(self, in_features, num_classes, hidden=512, drop=0.5):
        super().__init__()
        mid = hidden // 2
        self.head = nn.Sequential(
            nn.Linear(in_features, hidden), nn.BatchNorm1d(hidden),
            nn.ReLU(inplace=True), nn.Dropout(drop),
            nn.Linear(hidden, mid), nn.BatchNorm1d(mid),
            nn.ReLU(inplace=True), nn.Dropout(drop),
            nn.Linear(mid, num_classes))
    def forward(self, x):
        return self.head(x)

class MultiHeadNet(nn.Module):
    def __init__(self, arch, num_classes, drop, drop_path):
        super().__init__()
        self.backbone = timm.create_model(arch, pretrained=True, num_classes=0,
                                          drop_rate=drop, drop_path_rate=drop_path)
        self.classifier = ClassifierHead(self.backbone.num_features, num_classes,
                                         drop=drop)
    def forward(self, x):
        f = self.backbone.forward_features(x)
        if f.dim() == 4: f = f.mean(dim=[2, 3])
        elif f.dim() == 3: f = f.mean(dim=1)
        return self.classifier(f)

def build_model():
    if CFG["head"] == "plain":
        return timm.create_model(CFG["arch"], pretrained=True,
                                 num_classes=CFG["num_classes"],
                                 drop_rate=CFG["drop_rate"],
                                 drop_path_rate=CFG["drop_path_rate"])
    if CFG["head"] == "multihead":
        return MultiHeadNet(CFG["arch"], CFG["num_classes"],
                            CFG["drop_rate"], CFG["drop_path_rate"])
    raise ValueError("unknown head: " + CFG["head"])

model = build_model().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f"{CFG['arch']} / {CFG['head']} head — {n_params/1e6:.1f}M parameters")

efficientnet_b3 / plain head — 10.7M parameters


In [12]:
# ── Loss ─────────────────────────────────────────────────────────────────────
# Focal loss over softmax, with inverse-frequency class weights and an explicit
# extra multiplier on melanoma. The deployed model missed ~38% of melanomas and
# reported them as benign; weighting that error is the main lever available
# without collecting more data.
train_counts = np.array([counts[c] for c in CLASSES], dtype=np.float64)
class_weights = train_counts.sum() / (len(CLASSES) * train_counts)
class_weights = class_weights / class_weights.mean()
class_weights[MEL_IDX] *= CFG["mel_loss_weight"]
print("class weights:", {c: round(float(w), 3) for c, w in zip(CLASSES, class_weights)})
class_weights_t = torch.tensor(class_weights, dtype=torch.float32, device=DEVICE)

class FocalLoss(nn.Module):
    def __init__(self, weight, gamma, smoothing):
        super().__init__()
        self.weight, self.gamma, self.smoothing = weight, gamma, smoothing
    def forward(self, logits, target):
        logp = F.log_softmax(logits, dim=1)
        n = logits.size(1)
        with torch.no_grad():
            true = torch.zeros_like(logp).fill_(self.smoothing / (n - 1))
            true.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        ce = -(true * logp).sum(dim=1)
        pt = logp.gather(1, target.unsqueeze(1)).squeeze(1).exp()
        w = self.weight[target]
        return (w * (1 - pt).pow(self.gamma) * ce).mean()

criterion = FocalLoss(class_weights_t, CFG["focal_gamma"], CFG["label_smoothing"])

# Differential learning rates: the pretrained backbone is fine-tuned gently.
head_names = ["classifier", "fc", "head"]
head_params = [p for n, p in model.named_parameters()
               if any(h in n for h in head_names)]
backbone_params = [p for n, p in model.named_parameters()
                   if not any(h in n for h in head_names)]
optimizer = torch.optim.AdamW(
    [{"params": backbone_params, "lr": CFG["lr_backbone"]},
     {"params": head_params, "lr": CFG["lr_head"]}],
    weight_decay=CFG["weight_decay"])

steps = max(1, len(train_loader))
def lr_lambda(step):
    e = step / steps
    if e < CFG["warmup_epochs"]:
        return (e + 1e-8) / CFG["warmup_epochs"]
    p = (e - CFG["warmup_epochs"]) / max(1e-8, CFG["epochs"] - CFG["warmup_epochs"])
    return 0.5 * (1 + math.cos(math.pi * min(1.0, p)))
scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
scaler = torch.amp.GradScaler("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda")
print("optimizer and schedule ready")

class weights: {'akiec': 0.893, 'bcc': 0.564, 'bkl': 0.262, 'df': 2.916, 'mel': 0.66, 'nv': 0.044, 'vasc': 2.058}
optimizer and schedule ready


## 3. Training

Selection metric is macro-F1, with melanoma recall logged every epoch so a model
that trades melanoma away for overall accuracy is visible while it happens.

In [13]:
# ── Metric helpers ───────────────────────────────────────────────────────────
def per_class_scores(y_true, y_pred, n=len(CLASSES)):
    f1, rec, prec = [], [], []
    for i in range(n):
        tp = int(((y_pred == i) & (y_true == i)).sum())
        fp = int(((y_pred == i) & (y_true != i)).sum())
        fn = int(((y_pred != i) & (y_true == i)).sum())
        p = tp / (tp + fp) if tp + fp else 0.0
        r = tp / (tp + fn) if tp + fn else 0.0
        prec.append(p); rec.append(r)
        f1.append(2 * p * r / (p + r) if p + r else 0.0)
    return {"accuracy": float((y_pred == y_true).mean()),
            "macro_f1": float(np.mean(f1)),
            "precision": prec, "recall": rec, "f1": f1}

@torch.no_grad()
def collect_logits(net, loader):
    net.eval()
    outs, ys = [], []
    for x, y in loader:
        x = x.to(DEVICE, non_blocking=True)
        with torch.autocast("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda"):
            outs.append(net(x).float().cpu())
        ys.append(y)
    return torch.cat(outs).numpy(), torch.cat(ys).numpy()

In [14]:
# ── Train ────────────────────────────────────────────────────────────────────
best = {"macro_f1": -1.0, "epoch": -1}
history = []
patience = 0
BEST_PATH = f"{OUT_DIR}/_best_weights.pt"

for epoch in range(1, CFG["epochs"] + 1):
    model.train()
    running, seen, t0 = 0.0, 0, time.time()
    for x, y in train_loader:
        x = x.to(DEVICE, non_blocking=True); y = y.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda"):
            loss = criterion(model(x), y)
        scaler.scale(loss).backward()
        scaler.step(optimizer); scaler.update(); scheduler.step()
        running += loss.item() * x.size(0); seen += x.size(0)

    logits, y_true = collect_logits(model, val_loader)
    m = per_class_scores(y_true, logits.argmax(1))
    history.append({"epoch": epoch, "train_loss": running / max(1, seen),
                    "val_accuracy": m["accuracy"], "val_macro_f1": m["macro_f1"],
                    "val_mel_recall": m["recall"][MEL_IDX]})
    print(f"epoch {epoch:3d}  loss {running/max(1,seen):.4f}  "
          f"val acc {m['accuracy']:.4f}  macroF1 {m['macro_f1']:.4f}  "
          f"melRecall {m['recall'][MEL_IDX]:.4f}  ({time.time()-t0:.0f}s)")

    if m["macro_f1"] > best["macro_f1"]:
        best = {"macro_f1": m["macro_f1"], "epoch": epoch,
                "val_mel_recall": m["recall"][MEL_IDX]}
        torch.save(model.state_dict(), BEST_PATH)
        patience = 0
        print("            ^ best so far, checkpointed")
    else:
        patience += 1
        if patience >= CFG["early_stop_patience"]:
            print(f"early stopping: no improvement for {patience} epochs")
            break

pd.DataFrame(history).to_csv(f"{OUT_DIR}/training_history.csv", index=False)
model.load_state_dict(torch.load(BEST_PATH, map_location=DEVICE))
print(f"\nrestored best epoch {best['epoch']} (val macro-F1 {best['macro_f1']:.4f})")

/tmp/ipykernel_114/3569479948.py:16: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scaler.step(optimizer); scaler.update(); scheduler.step()


epoch   1  loss 2.8567  val acc 0.0818  macroF1 0.1078  melRecall 0.2176  (161s)
            ^ best so far, checkpointed
epoch   2  loss 1.0747  val acc 0.1329  macroF1 0.1687  melRecall 0.5118  (90s)
            ^ best so far, checkpointed
epoch   3  loss 0.6438  val acc 0.1780  macroF1 0.2077  melRecall 0.5647  (89s)
            ^ best so far, checkpointed
epoch   4  loss 0.4638  val acc 0.2408  macroF1 0.2808  melRecall 0.6294  (87s)
            ^ best so far, checkpointed
epoch   5  loss 0.3483  val acc 0.2814  macroF1 0.3015  melRecall 0.6176  (88s)
            ^ best so far, checkpointed
epoch   6  loss 0.2769  val acc 0.3887  macroF1 0.3431  melRecall 0.6118  (91s)
            ^ best so far, checkpointed
epoch   7  loss 0.2499  val acc 0.4208  macroF1 0.3815  melRecall 0.6941  (87s)
            ^ best so far, checkpointed
epoch   8  loss 0.2191  val acc 0.4732  macroF1 0.4173  melRecall 0.7294  (86s)
            ^ best so far, checkpointed
epoch   9  loss 0.1887  val acc 0.4490 

In [15]:
# ── Training curves (real ones, from the logged history) ─────────────────────
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

h = pd.DataFrame(history)
fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].plot(h.epoch, h.train_loss); ax[0].set_title("training loss")
ax[1].plot(h.epoch, h.val_macro_f1); ax[1].set_title("validation macro-F1")
ax[2].plot(h.epoch, h.val_mel_recall, color="#a8442a")
ax[2].set_title("validation melanoma recall")
for a in ax: a.set_xlabel("epoch"); a.grid(alpha=0.3)
fig.tight_layout(); fig.savefig(f"{OUT_DIR}/training_curves.png", dpi=150)
plt.close(fig)
print("training_curves.png written from measured history — keep this file; the "
      "previous project had no training log and its curves could not be reproduced")

training_curves.png written from measured history — keep this file; the previous project had no training log and its curves could not be reproduced


## 4. Calibration and thresholds — fitted on `calib`, never on `test`

Three things are fitted here, all on the calibration split:

1. **Temperature** — flattens over-confident probabilities.
2. **Per-class thresholds** — the decision rule serving uses.
3. **Melanoma alert threshold** — flags `p(mel)` above a cutoff regardless of
   which class wins, because melanoma misses still carry substantial `p(mel)`.

In [16]:
# ── Fit on the calibration split ─────────────────────────────────────────────
def to_probs(logits, temperature=1.0):
    z = torch.tensor(logits, dtype=torch.float32) / temperature
    if CFG["readout"] == "softmax":
        return torch.softmax(z, dim=1).numpy()
    return torch.sigmoid(z).numpy()

def decide(probs, thresholds):
    return np.argmax(probs - thresholds[None, :], axis=1)

def ece(conf, correct, bins=15):
    e = 0.0
    for i in range(bins):
        m = (conf > i / bins) & (conf <= (i + 1) / bins)
        if m.sum():
            e += m.mean() * abs(correct[m].mean() - conf[m].mean())
    return float(e)

cal_logits, cal_y = collect_logits(model, calib_loader)

# 1. temperature
best_T, best_e = 1.0, float("inf")
for T in np.arange(0.5, 6.01, 0.05):
    p = to_probs(cal_logits, T)
    yp = p.argmax(1)
    e = ece(p[np.arange(len(yp)), yp], yp == cal_y)
    if e < best_e:
        best_T, best_e = float(T), e
print(f"temperature      T = {best_T:.2f}   (calib ECE {best_e:.4f})")

# 2. per-class thresholds, greedy coordinate ascent on macro-F1
cal_probs = to_probs(cal_logits, best_T)
thresholds = np.full(len(CLASSES), 0.5)
def macro_f1_at(t):
    return per_class_scores(cal_y, decide(cal_probs, t))["macro_f1"]
current = macro_f1_at(thresholds)
for _ in range(6):
    improved = False
    for i in range(len(CLASSES)):
        for cand in np.round(np.arange(0.05, 0.96, 0.05), 2):
            trial = thresholds.copy(); trial[i] = cand
            v = macro_f1_at(trial)
            if v > current + 1e-9:
                current, thresholds, improved = v, trial, True
    if not improved:
        break
print("thresholds       ", {c: round(float(t), 2) for c, t in zip(CLASSES, thresholds)})
print(f"calib macro-F1   {current:.4f}")

# 3. melanoma alert threshold: lowest review rate reaching the target sensitivity
cal_pred = decide(cal_probs, thresholds)
mel_mask = cal_y == MEL_IDX
alert_t, alert_rate = None, 1.0
for t in np.round(np.arange(0.05, 0.96, 0.05), 2):
    surfaced = (cal_probs[:, MEL_IDX] >= t) | (cal_pred == MEL_IDX)
    if surfaced[mel_mask].mean() >= CFG["target_mel_sensitivity"]:
        rate = surfaced[~mel_mask].mean()
        if rate <= alert_rate:
            alert_t, alert_rate = float(t), float(rate)
if alert_t is None:
    alert_t = 0.50
    print("WARNING: target melanoma sensitivity unreachable on calib; using 0.50")
print(f"melanoma alert   p(mel) >= {alert_t:.2f}  (calib review rate {alert_rate:.3f})")

temperature      T = 2.10   (calib ECE 0.0366)
thresholds        {'akiec': 0.75, 'bcc': 0.65, 'bkl': 0.55, 'df': 0.7, 'mel': 0.75, 'nv': 0.35, 'vasc': 0.6}
calib macro-F1   0.6481
melanoma alert   p(mel) >= 0.25  (calib review rate 0.392)


## 5. Final evaluation — the test split, used exactly once

Nothing below feeds back into any fitting decision. If these numbers are
disappointing, the honest response is to change the training recipe and retrain,
not to tune against this split.

In [17]:
# ── Test-set evaluation ──────────────────────────────────────────────────────
test_logits, test_y = collect_logits(model, test_loader)
test_probs = to_probs(test_logits, best_T)
test_pred = decide(test_probs, thresholds)

m = per_class_scores(test_y, test_pred)
conf = test_probs[np.arange(len(test_pred)), test_pred]
mel_mask_t = test_y == MEL_IDX
surfaced = (test_probs[:, MEL_IDX] >= alert_t) | (test_pred == MEL_IDX)

print(f"test images            {len(test_y)}")
print(f"accuracy               {m['accuracy']:.4f}")
print(f"macro-F1               {m['macro_f1']:.4f}")
print(f"ECE                    {ece(conf, test_pred == test_y):.4f}")
print(f"melanoma recall        {m['recall'][MEL_IDX]:.4f}")
print(f"melanoma surfaced      {surfaced[mel_mask_t].mean():.4f}  (with alert channel)")
print(f"cases flagged          {surfaced[~mel_mask_t].mean():.4f}")
print()
print(f"{'class':<8}{'support':>9}{'prec':>9}{'recall':>9}{'f1':>9}")
for i, c in enumerate(CLASSES):
    print(f"{c:<8}{int((test_y==i).sum()):>9}{m['precision'][i]:>9.4f}"
          f"{m['recall'][i]:>9.4f}{m['f1'][i]:>9.4f}")

cm = np.zeros((len(CLASSES), len(CLASSES)), dtype=int)
for t, p in zip(test_y, test_pred):
    cm[t, p] += 1

fig, ax = plt.subplots(figsize=(8, 7))
norm = cm / np.maximum(cm.sum(1, keepdims=True), 1)
im = ax.imshow(norm, cmap="Blues", vmin=0, vmax=1)
ax.set_xticks(range(len(CLASSES)), CLASSES); ax.set_yticks(range(len(CLASSES)), CLASSES)
ax.set_xlabel("predicted"); ax.set_ylabel("true")
ax.set_title(f"Measured confusion matrix (n={len(test_y)})")
for i in range(len(CLASSES)):
    for j in range(len(CLASSES)):
        ax.text(j, i, f"{norm[i,j]:.2f}\n({cm[i,j]})", ha="center", va="center",
                fontsize=7, color="white" if norm[i, j] > 0.5 else "black")
fig.colorbar(im, ax=ax); fig.tight_layout()
fig.savefig(f"{OUT_DIR}/confusion_matrix_measured.png", dpi=150); plt.close(fig)
print("\nconfusion_matrix_measured.png written")

test images            1503
accuracy               0.7552
macro-F1               0.6416
ECE                    0.2367
melanoma recall        0.5706
melanoma surfaced      0.9235  (with alert channel)
cases flagged          0.4051

class     support     prec   recall       f1
akiec          52   0.5283   0.5385   0.5333
bcc            79   0.6818   0.7595   0.7186
bkl           158   0.5529   0.5949   0.5732
df             22   0.6429   0.4091   0.5000
mel           170   0.4491   0.5706   0.5026
nv           1000   0.8837   0.8280   0.8549
vasc           22   0.7600   0.8636   0.8085

confusion_matrix_measured.png written


In [ ]:
# ── Brightness robustness ────────────────────────────────────────────────────
# The deployed model changed its answer when an image was darkened. That biases
# against darker skin tones and poor lighting. This measures whether training
# augmentation actually fixed it. Darkening is a crude proxy for melanin, not a
# substitute for evaluating on a dataset with Fitzpatrick labels (e.g. DDI).
_MEAN = torch.tensor(CFG["norm_mean"]).view(1, 3, 1, 1)
_STD = torch.tensor(CFG["norm_std"]).view(1, 3, 1, 1)

@torch.no_grad()
def accuracy_at_brightness(factor):
    '''Scale brightness in pixel space, not in normalized space.'''
    model.eval()
    correct = total = 0
    for x, y in test_loader:
        pixels = (x * _STD + _MEAN).clamp(0, 1) * factor      # de-normalise, dim
        x = ((pixels.clamp(0, 1) - _MEAN) / _STD).to(DEVICE)  # re-normalise
        with torch.autocast("cuda", enabled=CFG["amp"] and DEVICE.type == "cuda"):
            p = model(x).float().cpu()
        correct += int((p.argmax(1) == y).sum()); total += len(y)
    return correct / total

print("brightness scale -> accuracy")
robust = {}
for f in [1.0, 0.75, 0.5, 0.35]:
    robust[f] = accuracy_at_brightness(f)
    print(f"  x{f:<5} {robust[f]:.4f}")
spread = max(robust.values()) - min(robust.values())
print(f"\nspread {spread:.4f}  ({'acceptable' if spread < 0.05 else 'STILL BRIGHTNESS-SENSITIVE'})")

## 6. Export — CPU-ready, self-describing

The bundle stores everything serving needs to reproduce training conditions, so
the architecture/resolution/readout mismatches that broke the previous
deployment cannot recur. Only tensors and plain primitives are saved, so
`torch.load(..., weights_only=True)` works on a CPU-only machine.

In [ ]:
# ── Export ───────────────────────────────────────────────────────────────────
import hashlib

# CPU, float32, no torch.compile / DataParallel prefixes, no numpy scalars.
state = {}
for k, v in model.state_dict().items():
    k = k.replace("_orig_mod.", "").replace("module.", "")
    state[k] = v.detach().cpu().float()

fingerprint = hashlib.sha256(
    b"".join(state[k].numpy().tobytes() for k in sorted(state))).hexdigest()[:16]

def py(x):
    '''numpy scalars break torch.load(weights_only=True); force plain floats.'''
    return float(x)

test_metrics = {
    "test_set_size": int(len(test_y)),
    "accuracy": py(m["accuracy"]),
    "macro_f1": py(m["macro_f1"]),
    "ece": py(ece(conf, test_pred == test_y)),
    "melanoma_recall": py(m["recall"][MEL_IDX]),
    "melanoma_surfaced": py(surfaced[mel_mask_t].mean()),
    "review_rate": py(surfaced[~mel_mask_t].mean()),
    "per_class": {c: {"support": int((test_y == i).sum()),
                      "precision": py(m["precision"][i]),
                      "recall": py(m["recall"][i]),
                      "f1": py(m["f1"][i])}
                  for i, c in enumerate(CLASSES)},
    "confusion_matrix": cm.tolist(),
}

bundle = {
    "format_version": 1,
    "created": datetime.now(timezone.utc).isoformat(),
    "arch": CFG["arch"],
    "head": CFG["head"],
    "num_classes": CFG["num_classes"],
    "classes": CLASSES,
    "img_size": CFG["img_size"],
    "norm_mean": CFG["norm_mean"],
    "norm_std": CFG["norm_std"],
    "readout": CFG["readout"],
    "decision_rule": "argmax(probability - class_threshold)",
    "temperature": py(best_T),
    "thresholds": {c: py(t) for c, t in zip(CLASSES, thresholds)},
    "mel_alert_threshold": py(alert_t),
    "weight_fingerprint": fingerprint,
    "best_epoch": int(best["epoch"]),
    "model_state_dict": state,
}
torch.save(bundle, f"{OUT_DIR}/dermascan_b3.pt")

# Backend-compatible sidecars, so this drops into the serving repo unchanged.
with open(f"{OUT_DIR}/class_thresholds.json", "w") as f:
    json.dump({"class_thresholds": {c: py(t) for c, t in zip(CLASSES, thresholds)},
               "readout": CFG["readout"],
               "fitted_on": "calibration split (lesion-disjoint)",
               "per_class_metrics": {
                   c: {"threshold": py(thresholds[i]),
                       "f1": py(m["f1"][i]), "precision": py(m["precision"][i]),
                       "recall": py(m["recall"][i])}
                   for i, c in enumerate(CLASSES)}}, f, indent=2)

with open(f"{OUT_DIR}/calibration.json", "w") as f:
    json.dump({"temperature": py(best_T), "mel_alert_threshold": py(alert_t),
               "readout": CFG["readout"],
               "target_sensitivity": CFG["target_mel_sensitivity"],
               "fitted_on": "calibration split (lesion-disjoint)",
               "reported_on": "held-out test split",
               "test_metrics_after": {
                   "accuracy": test_metrics["accuracy"],
                   "ece": test_metrics["ece"],
                   "melanoma_recall_argmax": test_metrics["melanoma_recall"],
                   "melanoma_surfaced": test_metrics["melanoma_surfaced"],
                   "review_rate": test_metrics["review_rate"]}}, f, indent=2)

with open(f"{OUT_DIR}/evaluation_results.json", "w") as f:
    json.dump(test_metrics, f, indent=2)
with open(f"{OUT_DIR}/train_config.json", "w") as f:
    json.dump(CFG, f, indent=2)

size_mb = os.path.getsize(f"{OUT_DIR}/dermascan_b3.pt") / 1024 / 1024
print(f"dermascan_b3.pt          {size_mb:.0f} MB   fingerprint {fingerprint}")
for f_ in ["class_thresholds.json", "calibration.json", "evaluation_results.json",
           "train_config.json", "training_history.csv", "training_curves.png",
           "confusion_matrix_measured.png",
           "split_train.csv", "split_val.csv", "split_calib.csv", "split_test.csv"]:
    print("   ", f_)

In [ ]:
# ── Verify the export loads on CPU, exactly as the server will load it ───────
# This is the check that would have caught the previous deployment's silent
# architecture mismatch. Do not skip it.
ck = torch.load(f"{OUT_DIR}/dermascan_b3.pt", map_location="cpu", weights_only=True)
print("weights_only load: OK (no numpy scalars in the bundle)")

cpu_model = (timm.create_model(ck["arch"], pretrained=False,
                               num_classes=ck["num_classes"])
             if ck["head"] == "plain"
             else MultiHeadNet(ck["arch"], ck["num_classes"], 0.0, 0.0))
cpu_model.load_state_dict(ck["model_state_dict"])   # strict
cpu_model.eval()
print("strict state_dict load on CPU: OK")

probe = torch.randn(1, 3, ck["img_size"], ck["img_size"])
with torch.no_grad():
    out = cpu_model(probe)
assert out.shape == (1, ck["num_classes"]), out.shape
print(f"CPU forward pass: OK  logits {tuple(out.shape)}")
print("classes:", ck["classes"])
print("img_size:", ck["img_size"], "| readout:", ck["readout"],
      "| T:", round(ck["temperature"], 3), "| mel alert:", ck["mel_alert_threshold"])

## 7. Deploying into the serving repository

Download from the notebook's **Output** panel, then in `MODEL_Skin-Cancer/`:

```bash
cp dermascan_b3.pt        models/latest.pt
cp class_thresholds.json  models/
cp calibration.json       models/
cp evaluation_results.json docs/
cp confusion_matrix_measured.png docs/
```

Then set the architecture to match what the bundle declares:

```bash
# .env
MODEL_ARCH=plain        # or multihead, matching CFG["head"]
IMG_SIZE=300            # must equal the bundle's img_size
```

`calibration.json` carries `readout`, so the backend picks it up automatically —
the softmax/sigmoid mismatch cannot recur silently.

Verify before trusting it:

```bash
python -m pytest
python scripts/evaluate_model.py --data-dir path/to/test_set
```

The evaluator's numbers should match `evaluation_results.json`. **If they do
not, the serving path and the training path have diverged again** — that
disagreement is precisely the bug this whole pipeline is built to surface, so
stop and find it rather than publishing either number.

### Honest expectations

Melanoma recall is bounded by how separable melanoma and nevus are at this
resolution with this much data. Class weighting and the alert channel shift the
operating point; they do not add information. If recall is still short of what
you need, the next levers are more melanoma data, higher input resolution, or a
dedicated melanoma-versus-nevus head — not further threshold tuning, which only
moves along the same curve.